# Phase 3 — Candidate Generation / Blocking Analysis
### Amazon ML Challenge: Multi-Source Entity Resolution

**Goal:** Drastically reduce the $O(N_1 \times (N_2 + N_3))$ search space down to a tractable candidate set using deterministic blocking rules, while maintaining high recall.

**Sources:**
- **S1 (Anchor):** `train_source1.tsv` / `test_source1.tsv`
- **S2:** `train_source2.tsv` / `test_source2.tsv`
- **S3:** `train_source3.tsv` / `test_source3.tsv`

**8 Deterministic Blocking Rules:**
1. `exact_name`: Exact normalized company name
2. `exact_address`: Exact normalized address string
3. `exact_postal`: Exact postal code / PIN
4. `country_first_name`: Country + first token of company name
5. `first_last_name`: First token + last token of company name
6. `country_first_last`: Country + first token + last token
7. `country_region_first`: Country + inferred region + first token
8. `postal_prefix_first`: 3-char postal prefix + first token

*Oversized block keys (> 5,000 entities) are pruned to prevent candidate explosion.*

In [ ]:
from __future__ import annotations

import csv
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Set, Tuple

# Auto-detect project paths (compatible with Local and Google Colab)
if Path("/content/AMAZON_ML_CHALLENGE-main").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE-main")
elif Path("/content/AMAZON_ML_CHALLENGE").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE")
elif Path("..").resolve().name == "AMAZON_ML_CHALLENGE":
    ROOT = Path("..").resolve()
else:
    ROOT = Path(".").resolve()

DATA = ROOT / "student_resource" / "dataset"
OUTPUT_DIR = ROOT / "data" / "processed" / "phase3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_BLOCK_SIZE = 5000

BLOCKS = [
    "exact_name",
    "exact_address",
    "exact_postal",
    "country_first_name",
    "first_last_name",
    "country_first_last",
    "country_region_first",
    "postal_prefix_first",
]

print(f"Project Root: {ROOT}")
print(f"Data Directory: {DATA} (Exists: {DATA.exists()})")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Configured Blocks: {BLOCKS}")

## Step 1: Text Normalization and Token Extractors
Standardizes text for blocking keys, extracts postal candidates, and isolates region components.

In [ ]:
def normalize_text(value: str) -> str:
    """Standardizes text for robust blocking key generation."""
    value = str(value or "").strip().lower()
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def postal_tokens(address: str) -> List[str]:
    """Extracts candidate postal code patterns from address string."""
    return re.findall(r"\b[a-z0-9][a-z0-9 -]{2,9}\b", address.lower())

def get_region(address: str) -> str:
    """Conservative region extraction: use the last comma-separated component before postal code."""
    parts = [p.strip() for p in address.split(",") if p.strip()]
    if len(parts) >= 2:
        return parts[-2]
    return ""

## Step 2: Block Key Generation Logic
Computes key signatures for the 8 deterministic blocking strategies.

In [ ]:
def block_key(row: Dict[str, str], block: str) -> str:
    """Generates a blocking key for a given row and blocking strategy."""
    name = normalize_text(row.get("business_name", ""))
    address = normalize_text(row.get("business_address", ""))
    country = normalize_text(row.get("country", ""))

    tokens = name.split()
    first = tokens[0] if tokens else ""
    last = tokens[-1] if tokens else ""

    postal = ""
    for tok in postal_tokens(address):
        if any(ch.isdigit() for ch in tok):
            postal = tok
            break

    if block == "exact_name":
        return name
    if block == "exact_address":
        return address
    if block == "exact_postal":
        return postal
    if block == "country_first_name":
        return f"{country}|{first}"
    if block == "first_last_name":
        return f"{first}|{last}"
    if block == "country_first_last":
        return f"{country}|{first}|{last}"
    if block == "country_region_first":
        region = normalize_text(get_region(address))
        return f"{country}|{region}|{first}"
    if block == "postal_prefix_first":
        prefix = re.sub(r"[^a-z0-9]", "", postal)[:3]
        return f"{prefix}|{first}"

    raise ValueError(f"Unknown block: {block}")

## Step 3: Inverted Blocking Index Builder
Indexes S2 and S3 entity IDs into buckets by blocking keys. Skips oversized buckets (`> MAX_BLOCK_SIZE`) to keep candidate counts scalable.

In [ ]:
def build_index(
    source_path: Path,
    block: str,
    max_block_size: int = MAX_BLOCK_SIZE,
) -> Dict[str, List[str]]:
    """Builds an inverted index mapping block_key -> list of entity_ids."""
    buckets: Dict[str, List[str]] = defaultdict(list)

    with source_path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            entity_id = row["entity_id"]
            key = block_key(row, block)
            if not key or key.endswith("|") or key.startswith("|"):
                continue

            bucket = buckets[key]
            if len(bucket) < max_block_size:
                bucket.append(entity_id)

    return dict(buckets)

## Step 4: Streaming Candidate Generation Pipeline
Streams Source 1 records, matches across the 8 blocking indexes, deduplicates candidate IDs, and streams out `candidate_pairs.tsv`.

In [ ]:
def generate_candidates(
    s1_path: Path,
    s2_path: Path,
    s3_path: Path,
    output_path: Path,
    max_block_size: int = MAX_BLOCK_SIZE,
) -> None:
    """
    Generate the final candidate_pairs.tsv.
    Output schema: source1_entity_id \t candidate_entity_ids
    """
    print("Building S2/S3 blocking indexes...")

    indexes: Dict[str, Dict[str, List[str]]] = {}
    for block in BLOCKS:
        print(f"  indexing {block}")
        idx2 = build_index(s2_path, block, max_block_size)
        idx3 = build_index(s3_path, block, max_block_size)

        merged: Dict[str, List[str]] = defaultdict(list)
        for key, ids in idx2.items():
            merged[key].extend(ids)
        for key, ids in idx3.items():
            merged[key].extend(ids)

        indexes[block] = dict(merged)
        print(f"    keys={len(indexes[block]):,}")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    total_pairs = 0
    total_s1 = 0

    with (
        s1_path.open("r", encoding="utf-8", errors="replace", newline="") as f1,
        output_path.open("w", encoding="utf-8", newline="") as out,
    ):
        reader = csv.DictReader(f1, delimiter="\t")
        writer = csv.writer(out, delimiter="\t")
        writer.writerow(["source1_entity_id", "candidate_entity_ids"])

        for row in reader:
            s1_id = row["entity_id"]
            candidates: Set[str] = set()

            for block in BLOCKS:
                key = block_key(row, block)
                if key in indexes[block]:
                    candidates.update(indexes[block][key])

            if candidates:
                writer.writerow([s1_id, ",".join(sorted(candidates))])
                total_pairs += len(candidates)

            total_s1 += 1
            if total_s1 % 100_000 == 0:
                print(
                    f"S1 processed={total_s1:,} | "
                    f"candidate links={total_pairs:,}"
                )

    print(f"Done. S1 rows={total_s1:,}")
    print(f"Candidate links={total_pairs:,}")
    print(f"Output={output_path}")

## Step 5: Candidate Pairs Analysis & Inspection
Analyzes candidate set volume, reduction ratio, and candidate distribution per S1 entity once `candidate_pairs.tsv` is loaded.

In [ ]:
def inspect_candidate_pairs(candidate_file: Path, max_display: int = 5):
    """Analyzes candidate pairs distribution and displays sample links."""
    if not candidate_file.exists():
        print(f"Candidate file not found at: {candidate_file}")
        print("Please provide or generate candidate_pairs.tsv first.")
        return
    
    print(f"Analyzing candidate pairs from: {candidate_file.name}...")
    total_s1 = 0
    total_candidates = 0
    sample_rows = []
    candidate_counts = []
    
    with open(candidate_file, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            s1_id = row["source1_entity_id"]
            cands = [c.strip() for c in row["candidate_entity_ids"].split(",") if c.strip()]
            count = len(cands)
            
            total_s1 += 1
            total_candidates += count
            candidate_counts.append(count)
            
            if len(sample_rows) < max_display:
                sample_rows.append((s1_id, cands[:5], count))
                
    avg_cands = total_candidates / max(1, total_s1)
    print(f"\n=== CANDIDATE GENERATION SUMMARY ===")
    print(f"- S1 Entities with Candidates : {total_s1:,}")
    print(f"- Total Candidate Pairs       : {total_candidates:,}")
    print(f"- Average Candidates / S1     : {avg_cands:.2f}")
    print("\nSample Candidate Links:")
    for s1_id, sample_cands, count in sample_rows:
        print(f"  {s1_id} -> {sample_cands} ... (total: {count})")

# File path definition
CANDIDATES_TSV = OUTPUT_DIR / "candidate_pairs.tsv"
# To inspect existing candidate pairs:
# inspect_candidate_pairs(CANDIDATES_TSV)